In [ ]:
import json
import re
import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()
def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "self-instruct").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Cannot find the project root")


def is_truthy(value):
    return str(value).strip().lower() in {"1", "true", "yes", "y"}


project_root = find_project_root()
legacy_mode = is_truthy(os.getenv("LEGACY_MODELS"))
print(os.getenv("LEGACY_MODELS"))
if legacy_mode:
    input_file_name = "cleaned_model_cards_legacy_step_3.jsonl"
    output_file_name = "cleaned_model_cards_legacy_step_4.jsonl"
else:
    input_file_name = "cleaned_model_cards_step_3.jsonl"
    output_file_name = "cleaned_model_cards_step_4.jsonl"

path_model_cards_generated = project_root / "self-instruct" / "data" / input_file_name

data = []
with open(path_model_cards_generated, "r") as f:
    data = json.load(f)
print(f"Loaded {len(data)} entries from {path_model_cards_generated}")

In [ ]:
data[:2]

In [ ]:
# Extract JSON object with {"model_description": "...text.."} from the response
count_failed = 0
count_no_json = 0
failed_entries = []  # Store failed entries
no_json_entries = []  # Store entries without JSON
cleaned_models = []

for entry in data:
    # Get the full response text
    description = str(entry.get("model_description") or "")

    if "</think>" in description:
        # Extract text after </think>
        text_after_think = description.split("</think>", 1)[-1].strip()
    else:
        text_after_think = description

    # Try to find JSON object containing "model_description" key
    # Look for pattern: {"model_description": "..."}
    match_json = re.search(
        r'\{[^}]*"model_description"[^}]*:.*?\}', text_after_think, re.DOTALL
    )

    if match_json:
        json_str = match_json.group(0).strip()
        try:
            # Try to parse the JSON
            parsed_json = json.loads(json_str)
            # Extract just the model_description value
            if "model_description" in parsed_json:
                entry["model_description"] = parsed_json["model_description"]
                cleaned_models.append(entry)
            else:
                entry["model_description"] = ""
                failed_entries.append(
                    {
                        "model_id": entry.get("model_id", "unknown"),
                        "reason": "JSON parsed but no 'model_description' key",
                        "raw_text": text_after_think,
                    }
                )
                count_failed += 1
        except json.JSONDecodeError as e:
            # If parsing fails, try to extract manually with regex
            manual_match = re.search(
                r'"model_description"\s*:\s*"([^"]*)"', text_after_think, re.DOTALL
            )
            if manual_match:
                entry["model_description"] = manual_match.group(1)
                cleaned_models.append(entry)
            else:
                entry["model_description"] = text_after_think
                failed_entries.append(
                    {
                        "model_id": entry.get("model_id", "unknown"),
                        "reason": f"JSON parse error: {str(e)}",
                        "raw_text": text_after_think,
                    }
                )
                count_failed += 1
    else:
        # No JSON found
        entry["model_description"] = ""
        no_json_entries.append(
            {
                "model_id": entry.get("model_id", "unknown"),
                "raw_text": text_after_think,
            }
        )
        count_no_json += 1


print(f"Extraction complete:")
print(f"  - Failed to parse JSON: {count_failed} entries")
print(f"  - No JSON found: {count_no_json} entries")
print(f"  - Successfully extracted: {len(cleaned_models)} entries")


print("\nFirst 3 model descriptions:")
for i, entry in enumerate(cleaned_models[:3], 1):
    desc = str(entry.get("model_description", ""))
    print(f"\n{i}. {entry['model_id']}:")
    print(f"   {desc}")


In [ ]:
# Print failed entries (JSON parsing errors)
print(f"=== FAILED ENTRIES ({len(failed_entries)}) ===\n")
for i, entry in enumerate(failed_entries[:10], 1):  # Show first 10
    print(f"{i}. Model: {entry['model_id']}")
    print(f"   Reason: {entry['reason']}")
    print(f"   Raw text: {entry['raw_text'][:200]}...")
    print()

if len(failed_entries) > 10:
    print(f"... and {len(failed_entries) - 10} more failed entries")

print(f"\n{'=' * 60}\n")

# Print entries without JSON
print(f"=== NO JSON FOUND ({len(no_json_entries)}) ===\n")
for i, entry in enumerate(no_json_entries[:10], 1):  # Show first 10
    print(f"{i}. Model: {entry['model_id']}")
    print(f"   Raw text: {entry['raw_text'][:200]}...")
    print()

if len(no_json_entries) > 10:
    print(f"... and {len(no_json_entries) - 10} more entries without JSON")

In [ ]:
# Remove model_description entries that are empty or None
print("Size before cleaning:", len(cleaned_models))
print(f"\nCleaning up entries with empty descriptions...")
cleaned_models = [
    entry
    for entry in cleaned_models
    if entry["model_description"] != "None" and entry["model_description"] != ""
]
print(f"After removing empty descriptions, {len(cleaned_models)} entries remain.")
# study len of model_description of each cleaned model with pandas, print the description with less then 500 characters

print("\nAnalyzing description lengths...")
lengths = [len(entry.get("model_description", "")) for entry in cleaned_models]
import pandas as pd

pd.Series(lengths).describe()

for entry in cleaned_models:
    description = entry.get("model_description", "")
    if len(description) < 500:
        print(description)
        print("-" * 80)
# remove models with description less than 500   characters
cleaned_models = [
    entry for entry in cleaned_models if len(entry.get("model_description", "")) >= 500
]
print(f"After filtering short descriptions, {len(cleaned_models)} entries remain.")

In [ ]:
# remove items that contain chinese characters
def contains_chinese(text):
    return any("\u4e00" <= char <= "\u9fff" for char in text)


print(f"\nRemoving entries with Chinese characters...")
print("Size before removing Chinese entries:", len(cleaned_models))
cleaned_models = [
    entry
    for entry in cleaned_models
    if not contains_chinese(entry.get("model_description", ""))
]
print("Size after removing Chinese entries:", len(cleaned_models))

In [ ]:
# print greater than 5000 characters
print("\nDescriptions longer than 5000 characters:")
for entry in cleaned_models:
    description = entry.get("model_description", "")
    if len(description) > 5000:
        print(f"Model ID: {entry['model_id']}, Length: {len(description)}")
        print(description)
        print("-" * 80)

In [ ]:
def remove_duplicates_by_modelcard(models, key="modelcard", priority_key="downloads"):
    """
    Remove duplicate models based on a key (e.g., 'modelcard').
    When duplicates are found, keep the one with the highest value for priority_key.

    Args:
        models: List of model dictionaries
        key: Field to check for duplicates (default: 'modelcard')
        priority_key: Field to use for selecting which duplicate to keep (default: 'downloads')

    Returns:
        List of unique models
    """
    unique_items = {}

    for item in models:
        key_value = item.get(key)

        if key_value is None:
            # Skip items without the key
            continue

        if key_value not in unique_items:
            # First time seeing this key value
            unique_items[key_value] = item
        else:
            # Duplicate found - keep the one with higher priority_key value
            current_priority = item.get(priority_key, 0)
            existing_priority = unique_items[key_value].get(priority_key, 0)

            if current_priority > existing_priority:
                unique_items[key_value] = item

    return list(unique_items.values())


# Check duplicates
model_cards = [item["modelcard"] for item in cleaned_models]
print(f"Unique model cards: {len(set(model_cards))} / {len(model_cards)}")

# Remove duplicates
cleaned_models = remove_duplicates_by_modelcard(
    cleaned_models, key="modelcard", priority_key="downloads"
)
print(f"After removing duplicates: {len(cleaned_models)} models")
cleaned_models = remove_duplicates_by_modelcard(
    cleaned_models,
    key="model_description",
    priority_key="downloads",
)
print(f"After removing duplicates: {len(cleaned_models)} models")

In [ ]:
# Save failed and no-json entries to files

# Save cleaned data in JSONL format
cleaned_file = project_root / "self-instruct" / "data" / output_file_name
# put model description in modelcard field
for model in cleaned_models:
    model["modelcard"] = model.pop("model_description", None)


print("sample of cleaned models:")
for model in cleaned_models[:3]:
    print(model["modelcard"])

# Write JSONL with proper formatting (no blank lines, no trailing newline)
with open(cleaned_file, "w") as f:
    for i, item in enumerate(cleaned_models):
        json_line = json.dumps(item, ensure_ascii=False)
        if i < len(cleaned_models) - 1:
            f.write(json_line + "\n")
        else:
            # Last line: no trailing newline to avoid empty line
            f.write(json_line)

print(f"Saved {len(cleaned_models)} cleaned entries to: {cleaned_file}")

In [ ]:
# Verify the saved JSONL file can be read correctly
data = []
with open(cleaned_file, "r") as f:
    for line in f:
        line = line.strip()
        if line:  # Skip empty lines
            data.append(json.loads(line))

print(f"Successfully loaded {len(data)} entries from {cleaned_file}")
print(f"\nFirst entry keys: {list(data[0].keys())}")
print(f"Sample modelcard length: {len(data[0].get('modelcard', ''))}")
